In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
diam = pd.read_csv('/content/diamonds.csv')
diam.head(5)

,Unnamed: 0,carat,cut,color,clarity,depth,table,price,x,y,z
0,1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [ ]:
diam.drop('Unnamed: 0', axis=1, inplace=True)

In [ ]:
diam.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53940 non-null  float64
 1   cut      53940 non-null  object 
 2   color    53940 non-null  object 
 3   clarity  53940 non-null  object 
 4   depth    53940 non-null  float64
 5   table    53940 non-null  float64
 6   price    53940 non-null  int64  
 7   x        53940 non-null  float64
 8   y        53940 non-null  float64
 9   z        53940 non-null  float64
dtypes: float64(6), int64(1), object(3)
memory usage: 4.1+ MB


In [ ]:
diam.duplicated().sum()

np.int64(146)

In [ ]:
diam.drop_duplicates(inplace=True)

In [ ]:
diam.head(5)

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [ ]:
diam['price'].value_counts(normalize=True)

,proportion
price,
605,0.002454
802,0.002342
625,0.002324
776,0.002305
828,0.002305
...,...
2736,0.000019
2727,0.000019
355,0.000019


In [ ]:
x = diam.drop('price', axis=1)
y = diam['price']

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler , OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [ ]:
cat_cols = x.select_dtypes(include='object').columns
num_cols = x.select_dtypes(exclude='object').columns

In [ ]:
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('cat', cat_pipe, cat_cols),
    ('num', num_pipe, num_cols)
])

In [ ]:
from imblearn.over_sampling import RandomOverSampler

In [ ]:
ros = RandomOverSampler(random_state=42)
x_train_resampled, y_train_resampled = ros.fit_resample(x_train, y_train)

In [ ]:
x_tr , x_val , y_tr , y_val = train_test_split(x_train_resampled, y_train_resampled, test_size=0.2, random_state=42)

In [ ]:
x_tr = preprocessor.fit_transform(x_tr)
x_val = preprocessor.transform(x_val)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
lin = LinearRegression()
lin.fit(x_tr, y_tr)

LinearRegression()

In [ ]:
lin.score(x_val, y_val)

0.9074747549537729

In [ ]:
print('MSE:', mean_squared_error(y_val, lin.predict(x_val)))

MSE: 2229533.68235654


In [ ]:
DTR = DecisionTreeRegressor()
DTR.fit(x_tr, y_tr)

DecisionTreeRegressor()

In [ ]:
DTR.score(x_val, y_val)

0.9999964293882176

In [ ]:
import joblib

joblib.dump(DTR, 'DTR.pkl')

['DTR.pkl']

In [ ]:
DTR_Grid = GridSearchCV(DTR,param_grid={'max_depth':range(1,10)},cv=5)
DTR_Grid.fit(x_tr, y_tr)

GridSearchCV(cv=5, estimator=DecisionTreeRegressor(),
             param_grid={'max_depth': range(1, 10)})

In [ ]:
DTR_Grid.score(x_val, y_val)

0.9508465502574099

In [ ]:
print('MSE:', mean_squared_error(y_val, DTR_Grid.predict(x_val)))

MSE: 1184425.631624878


In [ ]:
joblib.dump(DTR_Grid, 'DTR_Grid.pkl')

['DTR_Grid.pkl']

In [ ]:
import joblib
joblib.dump(preprocessor, 'preprocessor.pkl')

['preprocessor.pkl']

In [ ]:

diam.to_csv('diamond.csv', index=False)

In [ ]:
import joblib

# Save after training
joblib.dump(preprocessor, 'preprocessor.pkl')
joblib.dump(DTR_Grid, 'DTR_Grid.pkl')


['DTR_Grid.pkl']

In [ ]:
import sklearn
print(sklearn.__version__)

1.6.1


In [ ]:
import joblib
print(joblib.__version__)

1.5.1


In [ ]:
! pip install streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.1 MB/s eta 0:00:00


In [ ]:
import streamlit
print(streamlit.__version__)

1.47.0


In [ ]:
(pd.__version__)

'2.2.2'

In [ ]:
print(np.__version__)

2.0.2
